# T07: Fabric Lakehouse Quick Start

## What You'll Learn

Microsoft Fabric Lakehouses store data in two areas: **Files** (raw landing zone) and **Tables** (managed Delta Lake). In this notebook you will:

1. **Generate** a retail dataset at small scale
2. **Write** to Parquet files (simulating the Lakehouse Files section)
3. **Write** to Parquet files in a Tables directory (simulating the Lakehouse Tables section — Delta is Parquet on disk)
4. **Inspect** the file structure that was created
5. **Read back** the data to confirm round-trip fidelity

> **Note:** This notebook runs entirely locally. In a real Fabric Lakehouse you would use
> `spark.createDataFrame(df).write.format("delta")` to create managed Delta tables.
> Here we write Parquet files, which are the on-disk data layer of Delta Lake.

## Prerequisites

- Python 3.9 or later
- `pip install sqllocks-spindle` (uncomment the install cell below if needed)

## Time Estimate

**~5 minutes** from start to finish.

In [1]:
# Uncomment the line below if you're running in an environment
# where sqllocks-spindle is not yet installed.
# %pip install sqllocks-spindle

print("Installation cell ready. Uncomment the pip line above if needed.")

Installation cell ready. Uncomment the pip line above if needed.


## Generate Retail Data at Small Scale

### What we're about to do
We'll generate a complete retail dataset using the `small` scale. This gives us enough data to see realistic file sizes without waiting long.

### Why this matters
In a real Fabric Lakehouse workflow, you'd land raw files into the **Files** section and then promote curated data to **Tables** as Delta Lake. Spindle lets you simulate both paths locally before deploying to Fabric.

### What to expect
A summary showing every generated table and its row count.

In [2]:
from sqllocks_spindle import Spindle, RetailDomain

result = Spindle().generate(domain=RetailDomain(), scale="small", seed=42)

print(result.summary())

Spindle Generation Result
Schema: retail_3nf
Domain: retail
Mode:   3nf
Seed:   42
Time:   0.4s

Table                             Rows  Columns
---------------------------------------------
customer                         1,000        8
address                          1,500       10
product_category                    50        4
product                            500        6
promotion                          200        6
store                              150        5
order                            5,000        8
order_line                      12,500        8
return                             850        5
---------------------------------------------
TOTAL                           21,750


## Write to Parquet Files (Simulating Lakehouse Files)

### What we're about to do
We'll export the generated data as Parquet files into a directory structure that mirrors a Fabric Lakehouse **Files** landing zone: `Files/landing/retail/`.

### Why this matters
Parquet is the standard columnar format for analytics workloads. When data lands in the Files section of a Lakehouse, it's typically Parquet or CSV. Writing Parquet locally lets you test your ingestion pipelines before connecting to Fabric.

### What to expect
One Parquet file per table, written under `./lakehouse_demo/Files/landing/retail/`.

In [3]:
files = result.to_parquet("./lakehouse_demo/Files/landing/retail/")

print(f"Wrote {len(files)} Parquet files.")
for f in files:
    print(f"  {f}")

Wrote 9 Parquet files.
  lakehouse_demo\Files\landing\retail\customer.parquet
  lakehouse_demo\Files\landing\retail\address.parquet
  lakehouse_demo\Files\landing\retail\product_category.parquet
  lakehouse_demo\Files\landing\retail\product.parquet
  lakehouse_demo\Files\landing\retail\promotion.parquet
  lakehouse_demo\Files\landing\retail\store.parquet
  lakehouse_demo\Files\landing\retail\order.parquet
  lakehouse_demo\Files\landing\retail\order_line.parquet
  lakehouse_demo\Files\landing\retail\return.parquet


## Inspect the File Structure

### What we're about to do
We'll walk the `./lakehouse_demo` directory tree and display every Parquet file along with its size in bytes.

### Why this matters
Understanding file sizes and layout helps you estimate storage costs, plan partition strategies, and verify that your data landed where you expected.

### What to expect
A listing of all `.parquet` files with their relative paths and sizes.

In [4]:
from pathlib import Path

print("=== Lakehouse File Structure ===")
for f in sorted(Path("./lakehouse_demo").rglob("*.parquet")):
    print(f"  {f.relative_to('./lakehouse_demo')}: {f.stat().st_size:,} bytes")

=== Lakehouse File Structure ===
  Files\landing\retail\address.parquet: 98,170 bytes
  Files\landing\retail\customer.parquet: 40,606 bytes
  Files\landing\retail\order.parquet: 119,486 bytes
  Files\landing\retail\order_line.parquet: 184,761 bytes
  Files\landing\retail\product.parquet: 15,723 bytes
  Files\landing\retail\product_category.parquet: 3,981 bytes
  Files\landing\retail\promotion.parquet: 10,103 bytes
  Files\landing\retail\return.parquet: 23,704 bytes
  Files\landing\retail\store.parquet: 7,172 bytes
  Tables\retail\address.parquet: 98,170 bytes
  Tables\retail\customer.parquet: 40,606 bytes
  Tables\retail\order.parquet: 119,486 bytes
  Tables\retail\order_line.parquet: 184,761 bytes
  Tables\retail\product.parquet: 15,723 bytes
  Tables\retail\product_category.parquet: 3,981 bytes
  Tables\retail\promotion.parquet: 10,103 bytes
  Tables\retail\return.parquet: 23,704 bytes
  Tables\retail\store.parquet: 7,172 bytes


## Write to Delta-Compatible Parquet Files (Simulating Lakehouse Tables)

### What we're about to do
We'll export the same dataset as Parquet files into a directory structure that mirrors a Fabric Lakehouse **Tables** section. Delta Lake tables are Parquet files on disk plus a `_delta_log/` transaction log — so writing Parquet locally gives you the data layer of Delta.

### Why this matters
Fabric Lakehouse Tables are Delta Lake tables. Writing Parquet locally lets you verify data content and structure. In a real Fabric notebook, you'd convert to Spark DataFrames and write as managed Delta tables (see the commented Spark example below).

### What to expect
One Parquet file per table, written under `./lakehouse_demo/Tables/retail/`. A comment shows the equivalent Fabric Spark code.

In [5]:
# Write as Parquet files locally (the on-disk data layer of Delta Lake).
# In a Fabric notebook with Spark, you would instead do:
#   for table_name, df in result.tables.items():
#       spark.createDataFrame(df).write.format("delta").mode("overwrite") \
#           .option("overwriteSchema", "true").saveAsTable(f"retail_{table_name}")

from pathlib import Path

tables_path = "./lakehouse_demo/Tables/retail/"
table_files = result.to_parquet(tables_path)

print(f"Wrote {len(table_files)} Parquet table files (Delta-compatible).")
for f in table_files:
    print(f"  {f}")

# Show the structure for the Tables directory
tables_dir = Path(tables_path)
if tables_dir.exists():
    print(f"\n=== Tables directory structure ===")
    for item in sorted(tables_dir.rglob("*.parquet")):
        print(f"  {item.relative_to(tables_dir)}: {item.stat().st_size:,} bytes")

Wrote 9 Parquet table files (Delta-compatible).
  lakehouse_demo\Tables\retail\customer.parquet
  lakehouse_demo\Tables\retail\address.parquet
  lakehouse_demo\Tables\retail\product_category.parquet
  lakehouse_demo\Tables\retail\product.parquet
  lakehouse_demo\Tables\retail\promotion.parquet
  lakehouse_demo\Tables\retail\store.parquet
  lakehouse_demo\Tables\retail\order.parquet
  lakehouse_demo\Tables\retail\order_line.parquet
  lakehouse_demo\Tables\retail\return.parquet

=== Tables directory structure ===
  address.parquet: 98,170 bytes
  customer.parquet: 40,606 bytes
  order.parquet: 119,486 bytes
  order_line.parquet: 184,761 bytes
  product.parquet: 15,723 bytes
  product_category.parquet: 3,981 bytes
  promotion.parquet: 10,103 bytes
  return.parquet: 23,704 bytes
  store.parquet: 7,172 bytes


## Read Back the Data

### What we're about to do
We'll read the Parquet files back into DataFrames and compare row counts to the originals. This confirms that the write-read round trip preserves all data.

### Why this matters
Round-trip fidelity is essential. If you lose rows or corrupt data types during export, your downstream tests will be unreliable. This step proves the pipeline is lossless.

### What to expect
Matching row counts between the original in-memory data and the data read back from Parquet.

In [6]:
import pandas as pd
from pathlib import Path

print("=== Round-Trip Verification ===")
parquet_dir = Path("./lakehouse_demo/Files/landing/retail/")
for pf in sorted(parquet_dir.glob("*.parquet")):
    table_name = pf.stem
    df_read = pd.read_parquet(pf)
    original_rows = len(result.tables[table_name])
    read_rows = len(df_read)
    status = "MATCH" if original_rows == read_rows else "MISMATCH"
    print(f"  {table_name}: original={original_rows}, read_back={read_rows} [{status}]")

print("\nRound trip complete!")

=== Round-Trip Verification ===
  address: original=1500, read_back=1500 [MATCH]
  customer: original=1000, read_back=1000 [MATCH]
  order: original=5000, read_back=5000 [MATCH]
  order_line: original=12500, read_back=12500 [MATCH]
  product: original=500, read_back=500 [MATCH]
  product_category: original=50, read_back=50 [MATCH]
  promotion: original=200, read_back=200 [MATCH]
  return: original=850, read_back=850 [MATCH]
  store: original=150, read_back=150 [MATCH]

Round trip complete!


## What's Next?

You've simulated a complete Fabric Lakehouse landing — both Files (Parquet) and Tables (Delta Lake). Here's where to go from here:

| Notebook | What You'll Learn |
|----------|-------------------|
| **T08: Fabric SQL Database Quick Start** | Generate SQL INSERT scripts for Fabric SQL Database |
| **T09: Streaming Events** | Stream real-time events with configurable throughput and anomalies |
| **T11: Chaos Engineering** | Inject schema drift, null corruption, and referential breakage |

Happy generating!